In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q librosa soundfile openai-whisper joblib tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 6.3 MB/s eta 0:00:00


In [ ]:
import os
import json
import cv2
import math
import joblib
import librosa
import random
import warnings
import subprocess
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")

PROJECT_DIR = "/content/drive/MyDrive/Deepfake_Project"

RAW_DIR = os.path.join(PROJECT_DIR, "data/raw/dfdc_train_part_00")
PROCESSED_DIR = os.path.join(PROJECT_DIR, "data/processed")

CSV_DIR = os.path.join(PROCESSED_DIR, "csv")
FRAMES_DIR = os.path.join(PROCESSED_DIR, "frames")
AUDIO_DIR = os.path.join(PROCESSED_DIR, "audio")
TEXT_DIR = os.path.join(PROCESSED_DIR, "text")

OUTPUT_DIR = os.path.join(PROJECT_DIR, "data/outputs")
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
GRAPH_DIR = os.path.join(OUTPUT_DIR, "graphs")
REPORT_DIR = os.path.join(OUTPUT_DIR, "reports")
PRED_DIR = os.path.join(OUTPUT_DIR, "predictions")
EXPLAIN_DIR = os.path.join(OUTPUT_DIR, "explainability")
UNCERTAIN_DIR = os.path.join(OUTPUT_DIR, "uncertain_samples")

for folder in [
    CSV_DIR, FRAMES_DIR, AUDIO_DIR, TEXT_DIR,
    MODEL_DIR, GRAPH_DIR, REPORT_DIR, PRED_DIR,
    EXPLAIN_DIR, UNCERTAIN_DIR
]:
    os.makedirs(folder, exist_ok=True)

print("All folders are ready.")

All folders are ready.


In [ ]:
metadata_path = os.path.join(RAW_DIR, "metadata.json")

with open(metadata_path, "r") as f:
    metadata = json.load(f)

records = []

for filename, info in metadata.items():
    video_path = os.path.join(RAW_DIR, filename)

    if os.path.exists(video_path):
        label = info["label"]  # REAL or FAKE
        target = 0 if label == "FAKE" else 1

        records.append({
            "filename": filename,
            "video_path": video_path,
            "label": label,
            "target": target,
            "original": info.get("original", None)
        })

df = pd.DataFrame(records)
df.to_csv(os.path.join(CSV_DIR, "dfdc_metadata.csv"), index=False)

print("Total videos found:", len(df))
print(df["label"].value_counts())
df.head()

Total videos found: 1334
label
FAKE    1248
REAL      86
Name: count, dtype: int64


,filename,video_path,label,target,original
0,owxbbpjpch.mp4,/content/drive/MyDrive/Deepfake_Project/data/r...,FAKE,0,wynotylpnm.mp4
1,vpmyeepbep.mp4,/content/drive/MyDrive/Deepfake_Project/data/r...,REAL,1,None
2,fzvpbrzssi.mp4,/content/drive/MyDrive/Deepfake_Project/data/r...,REAL,1,None
3,htorvhbcae.mp4,/content/drive/MyDrive/Deepfake_Project/data/r...,FAKE,0,wclvkepakb.mp4
4,fckxaqjbxk.mp4,/content/drive/MyDrive/Deepfake_Project/data/r...,FAKE,0,vpmyeepbep.mp4


In [ ]:
SAMPLE_PER_CLASS = 80  # change to 300, 500, 1000 later

df_sample = (
    df.groupby("target", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), SAMPLE_PER_CLASS), random_state=42))
    .reset_index(drop=True)
)

train_df, test_df = train_test_split(
    df_sample,
    test_size=0.2,
    random_state=42,
    stratify=df_sample["target"]
)

train_df.to_csv(os.path.join(CSV_DIR, "train.csv"), index=False)
test_df.to_csv(os.path.join(CSV_DIR, "test.csv"), index=False)

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("Train labels:")
print(train_df["label"].value_counts())
print("Test labels:")
print(test_df["label"].value_counts())

Train size: 128
Test size: 32
Train labels:
label
FAKE    64
REAL    64
Name: count, dtype: int64
Test labels:
label
REAL    16
FAKE    16
Name: count, dtype: int64


In [ ]:
NUM_FRAMES = 5  # Reduced from 10 to 5 to mitigate potential memory issues

def extract_frames(video_path, output_folder, num_frames=5):
    os.makedirs(output_folder, exist_ok=True)

    existing_frames = list(Path(output_folder).glob("*.jpg"))
    if len(existing_frames) >= num_frames:
        return True

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return False

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return False

    frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    saved = 0

    for idx, frame_no in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
        ret, frame = cap.read()

        if ret:
            frame = cv2.resize(frame, (224, 224))
            frame_path = os.path.join(output_folder, f"frame_{idx:03d}.jpg")
            cv2.imwrite(frame_path, frame)
            saved += 1

    cap.release()
    return saved > 0


all_df = pd.concat([train_df, test_df]).reset_index(drop=True)

for _, row in tqdm(all_df.iterrows(), total=len(all_df)):
    video_id = Path(row["filename"]).stem
    output_folder = os.path.join(FRAMES_DIR, video_id)
    extract_frames(row["video_path"], output_folder, NUM_FRAMES)

print("Frame extraction completed.")

100%|██████████| 160/160 [00:15<00:00, 10.64it/s]

Frame extraction completed.


In [ ]:
def extract_audio(video_path, audio_path):
    if os.path.exists(audio_path):
        return True

    command = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        audio_path
    ]

    try:
        subprocess.run(
            command,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=60
        )
        return os.path.exists(audio_path)
    except:
        return False


for _, row in tqdm(all_df.iterrows(), total=len(all_df)):
    video_id = Path(row["filename"]).stem
    audio_path = os.path.join(AUDIO_DIR, f"{video_id}.wav")
    extract_audio(row["video_path"], audio_path)

print("Audio extraction completed.")

100%|██████████| 160/160 [00:00<00:00, 183.65it/s]

Audio extraction completed.


In [ ]:
visual_feature_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(224, 224, 3)
)

def extract_visual_feature(video_id):
    frame_folder = os.path.join(FRAMES_DIR, video_id)
    frame_paths = sorted(list(Path(frame_folder).glob("*.jpg")))

    if len(frame_paths) == 0:
        return np.zeros(1280)

    imgs = []

    for frame_path in frame_paths:
        img = image.load_img(frame_path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        imgs.append(img_array)

    imgs = np.array(imgs)

    # Explicitly add a batch dimension if it's a single 3D image (H, W, C) -> (1, H, W, C)
    if imgs.ndim == 3:
        imgs = np.expand_dims(imgs, axis=0)

    imgs = preprocess_input(imgs)

    features = visual_feature_model.predict(imgs, verbose=0)
    video_feature = np.mean(features, axis=0)

    return video_feature


visual_records = []

for _, row in tqdm(all_df.iterrows(), total=len(all_df)):
    filename = row["filename"]
    video_id = Path(filename).stem

    feature = extract_visual_feature(video_id)

    record = {"filename": filename}
    for i, value in enumerate(feature):
        record[f"v_{i}"] = value

    visual_records.append(record)

visual_df = pd.DataFrame(visual_records)
visual_df.to_csv(os.path.join(CSV_DIR, "visual_features.csv"), index=False)

print("Visual features saved.")
visual_df.head()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


100%|██████████| 160/160 [17:00<00:00,  6.38s/it]


Visual features saved.


,filename,v_0,v_1,v_2,v_3,v_4,v_5,v_6,v_7,v_8,...,v_1270,v_1271,v_1272,v_1273,v_1274,v_1275,v_1276,v_1277,v_1278,v_1279
0,uhakqelqri.mp4,0.029131,0.011701,1.149107,0.907663,0.522636,0.419046,0.564169,0.044872,0.084233,...,0.077604,0.329895,0.354589,1.475502,0.050973,0.853795,0.237182,0.707643,0.000000,0.768487
1,acdkfksyev.mp4,0.000000,0.000000,0.372378,0.160167,0.236375,0.816679,0.118553,0.001949,0.318956,...,0.293554,0.598263,0.194276,0.032810,0.007167,0.171869,0.531720,1.630385,0.770408,0.065029
2,qlueobbhbc.mp4,0.000000,0.008715,0.181905,1.573426,0.960257,0.451997,0.237549,0.280989,0.066560,...,0.033225,0.283410,0.354332,1.459398,0.195084,0.236280,0.165543,0.164111,0.058603,0.610832
3,zvsivkdkak.mp4,0.000000,0.011508,1.056725,0.616760,0.222080,0.222271,0.325355,0.431152,0.040972,...,0.069516,0.609879,0.273408,0.738330,0.130240,0.024756,0.196438,1.103394,0.029898,0.511721
4,yeouperxzc.mp4,0.000000,0.009459,1.379502,1.054801,0.200784,0.375505,0.140581,0.243007,0.062780,...,0.088110,0.985281,0.336539,0.680582,0.127608,0.016389,0.121209,1.266871,0.008483,0.264118


In [ ]:
def extract_audio_feature(video_id, n_mfcc=40):
    audio_path = os.path.join(AUDIO_DIR, f"{video_id}.wav")

    if not os.path.exists(audio_path):
        return np.zeros(n_mfcc * 6)

    try:
        y, sr = librosa.load(audio_path, sr=16000)

        if len(y) == 0:
            return np.zeros(n_mfcc * 6)

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        delta = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)

        combined = np.vstack([mfcc, delta, delta2])

        mean_features = np.mean(combined, axis=1)
        std_features = np.std(combined, axis=1)

        final_audio_feature = np.concatenate([mean_features, std_features])
        return final_audio_feature

    except:
        return np.zeros(n_mfcc * 6)


audio_records = []

for _, row in tqdm(all_df.iterrows(), total=len(all_df)):
    filename = row["filename"]
    video_id = Path(filename).stem

    feature = extract_audio_feature(video_id)

    record = {"filename": filename}
    for i, value in enumerate(feature):
        record[f"a_{i}"] = value

    audio_records.append(record)

audio_df = pd.DataFrame(audio_records)
audio_df.to_csv(os.path.join(CSV_DIR, "audio_features.csv"), index=False)

print("Audio features saved.")
audio_df.head()

100%|██████████| 160/160 [01:26<00:00,  1.85it/s]


Audio features saved.


,filename,a_0,a_1,a_2,a_3,a_4,a_5,a_6,a_7,a_8,...,a_230,a_231,a_232,a_233,a_234,a_235,a_236,a_237,a_238,a_239
0,uhakqelqri.mp4,-463.553802,108.214333,29.269228,29.096596,3.263124,6.542248,-6.951723,-3.940777,-6.702765,...,0.490800,0.403834,0.506226,0.565316,0.473974,0.502403,0.446329,0.454535,0.473252,0.595706
1,acdkfksyev.mp4,-466.448547,93.671402,20.075800,27.386126,-5.711513,-5.955157,-20.271883,-8.408575,-16.389904,...,0.642629,0.479350,0.518739,0.622110,0.677712,0.628149,0.480156,0.554573,0.522670,0.510666
2,qlueobbhbc.mp4,-414.830719,77.876434,3.840157,19.869804,-15.311216,-8.846332,-13.155747,-2.533703,-16.452341,...,0.749057,0.868719,0.825038,0.760619,0.832835,0.830727,0.900197,0.908264,0.965117,0.882321
3,zvsivkdkak.mp4,-454.562164,65.920738,21.588097,24.630316,-8.208398,-3.089886,-13.455689,2.218719,-8.474242,...,0.870857,1.034640,1.059232,0.958884,0.996586,1.038596,0.989128,1.096072,0.938198,0.923706
4,yeouperxzc.mp4,-450.370819,70.314323,19.556286,24.422426,-8.084851,-2.875438,-10.360196,0.044960,-8.741209,...,0.731818,0.870300,0.876890,0.966066,0.936093,1.090780,0.914200,0.911747,0.969128,0.917063


In [ ]:
import whisper

whisper_model = whisper.load_model("tiny")

def transcribe_audio(video_id):
    audio_path = os.path.join(AUDIO_DIR, f"{video_id}.wav")
    text_path = os.path.join(TEXT_DIR, f"{video_id}.txt")

    if os.path.exists(text_path):
        with open(text_path, "r", encoding="utf-8") as f:
            return f.read()

    if not os.path.exists(audio_path):
        transcript = ""
    else:
        try:
            result = whisper_model.transcribe(audio_path)
            transcript = result["text"]
        except:
            transcript = ""

    with open(text_path, "w", encoding="utf-8") as f:
        f.write(transcript)

    return transcript


text_records = []

for _, row in tqdm(all_df.iterrows(), total=len(all_df)):
    filename = row["filename"]
    video_id = Path(filename).stem

    transcript = transcribe_audio(video_id)

    text_records.append({
        "filename": filename,
        "transcript": transcript
    })

text_df = pd.DataFrame(text_records)
text_df.to_csv(os.path.join(CSV_DIR, "text_transcripts.csv"), index=False)

print("Text extraction completed.")
text_df.head()

100%|██████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 276MiB/s]
100%|██████████| 160/160 [00:49<00:00,  3.23it/s]


Text extraction completed.


,filename,transcript
0,uhakqelqri.mp4,make something beautiful through those aforem...
1,acdkfksyev.mp4,There's nobody's in a field guilty for killin...
2,qlueobbhbc.mp4,"you could have 40, 50 kids in a classroom and..."
3,zvsivkdkak.mp4,understand that what the other person has to ...
4,yeouperxzc.mp4,of what is important to us. So I do think tha...


In [ ]:
train_df = pd.read_csv(os.path.join(CSV_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(CSV_DIR, "test.csv"))

visual_df = pd.read_csv(os.path.join(CSV_DIR, "visual_features.csv")).set_index("filename")
audio_df = pd.read_csv(os.path.join(CSV_DIR, "audio_features.csv")).set_index("filename")
text_df = pd.read_csv(os.path.join(CSV_DIR, "text_transcripts.csv")).set_index("filename")

visual_cols = [col for col in visual_df.columns if col.startswith("v_")]
audio_cols = [col for col in audio_df.columns if col.startswith("a_")]

def get_text(filename):
    try:
        text = text_df.loc[filename, "transcript"]
        if pd.isna(text):
            return ""
        return str(text)
    except:
        return ""

train_files = train_df["filename"].tolist()
test_files = test_df["filename"].tolist()

X_visual_train = visual_df.reindex(train_files).fillna(0)[visual_cols].values
X_visual_test = visual_df.reindex(test_files).fillna(0)[visual_cols].values

X_audio_train = audio_df.reindex(train_files).fillna(0)[audio_cols].values
X_audio_test = audio_df.reindex(test_files).fillna(0)[audio_cols].values

train_texts = [get_text(f) for f in train_files]
test_texts = [get_text(f) for f in test_files]

tfidf = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X_text_train = tfidf.fit_transform(train_texts).toarray()
X_text_test = tfidf.transform(test_texts).toarray()

X_train = np.hstack([X_visual_train, X_audio_train, X_text_train])
X_test = np.hstack([X_visual_test, X_audio_test, X_text_test])

y_train = train_df["target"].values
y_test = test_df["target"].values

feature_names = (
    visual_cols +
    audio_cols +
    [f"text_{word}" for word in tfidf.get_feature_names_out()]
)

# --- Feature Combination (Interaction Features) ---
# Example: Create an interaction feature between the first visual feature (v_0) and the first audio feature (a_0)
# Ensure indices are correct. v_0 is the first visual feature, a_0 is the first audio feature after visual.

# Find index for 'v_0'
v0_idx = feature_names.index('v_0') if 'v_0' in feature_names else -1

# Find index for 'a_0'
a0_idx = feature_names.index('a_0') if 'a_0' in feature_names else -1

# Find index for 'text_the'
text_the_idx = feature_names.index('text_the') if 'text_the' in feature_names else -1

new_features_train = []
new_features_test = []
new_feature_names = []

if v0_idx != -1 and a0_idx != -1:
    interaction_v0_a0_train = X_train[:, v0_idx] * X_train[:, a0_idx]
    interaction_v0_a0_test = X_test[:, v0_idx] * X_test[:, a0_idx]
    new_features_train.append(interaction_v0_a0_train.reshape(-1, 1))
    new_features_test.append(interaction_v0_a0_test.reshape(-1, 1))
    new_feature_names.append("v0_x_a0")

if v0_idx != -1 and text_the_idx != -1:
    interaction_v0_text_the_train = X_train[:, v0_idx] * X_train[:, text_the_idx]
    interaction_v0_text_the_test = X_test[:, v0_idx] * X_test[:, text_the_idx]
    new_features_train.append(interaction_v0_text_the_train.reshape(-1, 1))
    new_features_test.append(interaction_v0_text_the_test.reshape(-1, 1))
    new_feature_names.append("v0_x_text_the")

# Append new features if any were created
if new_features_train:
    X_train = np.hstack([X_train] + new_features_train)
    X_test = np.hstack([X_test] + new_features_test)
    feature_names.extend(new_feature_names)
# --- End Feature Combination ---

print("Fused train shape:", X_train.shape)
print("Fused test shape:", X_test.shape)


Fused train shape: (128, 2022)
Fused test shape: (32, 2022)


In [ ]:
train_df = pd.read_csv(os.path.join(CSV_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(CSV_DIR, "test.csv"))

# Create DataFrame for training fused features
train_fused_df = pd.DataFrame(X_train, columns=feature_names)
train_fused_df["filename"] = train_df["filename"].values
train_fused_df["target"] = y_train

# Create DataFrame for testing fused features
test_fused_df = pd.DataFrame(X_test, columns=feature_names)
test_fused_df["filename"] = test_df["filename"].values
test_fused_df["target"] = y_test

# Concatenate train and test fused features
fused_features_df = pd.concat([train_fused_df, test_fused_df], ignore_index=True)

# Save to CSV
fused_features_csv_path = os.path.join(CSV_DIR, "fused_features.csv")
fused_features_df.to_csv(fused_features_csv_path, index=False)

print(f"Fused features saved to: {fused_features_csv_path}")
print("Shape of fused features DataFrame:", fused_features_df.shape)
display(fused_features_df.head())

Fused features saved to: /content/drive/MyDrive/Deepfake_Project/data/processed/csv/fused_features.csv
Shape of fused features DataFrame: (160, 2024)


,v_0,v_1,v_2,v_3,v_4,v_5,v_6,v_7,v_8,v_9,...,text_yourself,text_zombie,text_zombie apocalypse,text_zombies,text_zombies and,text_zombies it,v0_x_a0,v0_x_text_the,filename,target
0,0.029131,0.011701,1.149107,0.907663,0.522636,0.419046,0.564169,0.044872,0.084233,0.119583,...,0.000000,0.0,0.0,0.000000,0.0,0.000000,-13.50357,0.0,uhakqelqri.mp4,0
1,0.000000,0.000000,0.372378,0.160167,0.236375,0.816679,0.118553,0.001949,0.318956,0.478704,...,0.142379,0.0,0.0,0.113053,0.0,0.142379,-0.00000,0.0,acdkfksyev.mp4,0
2,0.000000,0.008715,0.181905,1.573426,0.960257,0.451997,0.237549,0.280989,0.066560,0.008860,...,0.000000,0.0,0.0,0.000000,0.0,0.000000,-0.00000,0.0,qlueobbhbc.mp4,0
3,0.000000,0.011508,1.056725,0.616760,0.222080,0.222271,0.325355,0.431152,0.040972,0.001354,...,0.000000,0.0,0.0,0.000000,0.0,0.000000,-0.00000,0.0,zvsivkdkak.mp4,0
4,0.000000,0.009459,1.379502,1.054801,0.200784,0.375505,0.140581,0.243007,0.062780,0.002823,...,0.000000,0.0,0.0,0.000000,0.0,0.000000,-0.00000,0.0,yeouperxzc.mp4,0


## Model Training and Evaluation

We will now train a Support Vector Classifier (SVC) using the fused features. First, we'll scale the features, skipping explicit feature selection as requested, and then train the model.

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled.")

Features scaled.


With scaled features (and skipping explicit feature selection as per your request), we can now train our SVC model and make predictions.

In [ ]:
# Train a Support Vector Classifier
# Using a linear kernel for simplicity and faster training
model = SVC(kernel='linear', random_state=42, probability=True)
model.fit(X_train_scaled, y_train)

print("SVC model trained.")

# Make predictions
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("Predictions made.")

SVC model trained.
Predictions made.


Finally, let's evaluate the performance of our classification model using various metrics.

In [ ]:
# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
display(pd.DataFrame(cm, index=['Actual FAKE', 'Actual REAL'], columns=['Predicted FAKE', 'Predicted REAL']))

Accuracy: 0.5000
Precision: 0.5000
Recall: 0.3750
F1 Score: 0.4286

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.62      0.56        16
           1       0.50      0.38      0.43        16

    accuracy                           0.50        32
   macro avg       0.50      0.50      0.49        32
weighted avg       0.50      0.50      0.49        32


Confusion Matrix:


,Predicted FAKE,Predicted REAL
Actual FAKE,10,6
Actual REAL,10,6


## Gaussian Naive Bayes Classifier

Let's apply a simple probabilistic model, Gaussian Naive Bayes, to our features.

In [ ]:
from sklearn.naive_bayes import GaussianNB

# Initialize and train the Gaussian Naive Bayes model
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

print("Gaussian Naive Bayes model trained.")

# Make predictions
y_pred_nb = nb_model.predict(X_test_scaled)

# Evaluate the model
accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test, y_pred_nb)
recall_nb = recall_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)

print(f"\n--- Gaussian Naive Bayes Performance ---")
print(f"Accuracy: {accuracy_nb:.4f}")
print(f"Precision: {precision_nb:.4f}")
print(f"Recall: {recall_nb:.4f}")
print(f"F1 Score: {f1_nb:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

print("\nConfusion Matrix:")
cm_nb = confusion_matrix(y_test, y_pred_nb)
display(pd.DataFrame(cm_nb, index=['Actual FAKE', 'Actual REAL'], columns=['Predicted FAKE', 'Predicted REAL']))

Gaussian Naive Bayes model trained.

--- Gaussian Naive Bayes Performance ---
Accuracy: 0.6250
Precision: 0.5833
Recall: 0.8750
F1 Score: 0.7000

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.38      0.50        16
           1       0.58      0.88      0.70        16

    accuracy                           0.62        32
   macro avg       0.67      0.62      0.60        32
weighted avg       0.67      0.62      0.60        32


Confusion Matrix:


,Predicted FAKE,Predicted REAL
Actual FAKE,6,10
Actual REAL,2,14


## Convolutional Neural Network (CNN) for Fused Features

For a CNN, we will treat the feature vector as a 1D sequence and use `Conv1D` layers. This can help in learning local patterns within the concatenated feature vector.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

# Reshape data for CNN (add a channel dimension)
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

# Build the 1D CNN model
cnn_model = Sequential([
    Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid') # Sigmoid for binary classification
])

cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("CNN model built and compiled.")
cnn_model.summary()

CNN model built and compiled.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 2020, 32)       │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 1010, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1010, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 1008, 64)       │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 504, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 504, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32256)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     2,064,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,070,849 (7.90 MB)

 Trainable params: 2,070,849 (7.90 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train the CNN model
# Using a smaller batch size and fewer epochs for demonstration, consider adjusting for better performance
history_cnn = cnn_model.fit(
    X_train_cnn, y_train,
    epochs=10, # Reduced epochs for faster execution
    batch_size=32,
    validation_data=(X_test_cnn, y_test),
    verbose=1
)

print("CNN model trained.")

# Evaluate the CNN model
y_pred_cnn_prob = cnn_model.predict(X_test_cnn)
y_pred_cnn = (y_pred_cnn_prob > 0.5).astype(int)

accuracy_cnn = accuracy_score(y_test, y_pred_cnn)
precision_cnn = precision_score(y_test, y_pred_cnn)
recall_cnn = recall_score(y_test, y_pred_cnn)
f1_cnn = f1_score(y_test, y_pred_cnn)

print(f"\n--- CNN Performance ---")
print(f"Accuracy: {accuracy_cnn:.4f}")
print(f"Precision: {precision_cnn:.4f}")
print(f"Recall: {recall_cnn:.4f}")
print(f"F1 Score: {f1_cnn:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_cnn))

print("\nConfusion Matrix:")
cm_cnn = confusion_matrix(y_test, y_pred_cnn)
display(pd.DataFrame(cm_cnn, index=['Actual FAKE', 'Actual REAL'], columns=['Predicted FAKE', 'Predicted REAL']))

Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 228ms/step - accuracy: 0.4922 - loss: 1.3756 - val_accuracy: 0.5000 - val_loss: 0.9011
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 146ms/step - accuracy: 0.5156 - loss: 0.9050 - val_accuracy: 0.6562 - val_loss: 0.6545
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 153ms/step - accuracy: 0.5938 - loss: 0.7578 - val_accuracy: 0.5938 - val_loss: 0.6405
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 141ms/step - accuracy: 0.6797 - loss: 0.6237 - val_accuracy: 0.5625 - val_loss: 0.6322
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 210ms/step - accuracy: 0.7188 - loss: 0.5943 - val_accuracy: 0.5625 - val_loss: 0.6522
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 268ms/step - accuracy: 0.6875 - loss: 0.5763 - val_accuracy: 0.5938 - val_loss: 0.6653
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 238ms/step - accuracy: 0.7188 - loss: 0.5654 - val_accuracy: 0.6875 - val_loss: 0.6611
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 248ms/step - accuracy: 0.7031 - loss: 0.5829 - val_accuracy: 0.6875 - val_loss:

,Predicted FAKE,Predicted REAL
Actual FAKE,10,6
Actual REAL,6,10


## Long Short-Term Memory (LSTM) Network for Fused Features

For an LSTM, we will also treat the feature vector as a sequence. Although the fused features are not inherently temporal, an LSTM can still capture dependencies across the feature vector elements.

In [ ]:
from tensorflow.keras.layers import LSTM

# Reshape data for LSTM (samples, timesteps, features). Here, timesteps = n_features, features = 1
X_train_lstm = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_lstm = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

# Build the LSTM model
lstm_model = Sequential([
    LSTM(64, activation='relu', input_shape=(X_train_lstm.shape[1], 1), return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid') # Sigmoid for binary classification
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("LSTM model built and compiled.")
lstm_model.summary()

LSTM model built and compiled.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,009 (74.25 KB)

 Trainable params: 19,009 (74.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train the LSTM model
history_lstm = lstm_model.fit(
    X_train_lstm, y_train,
    epochs=10, # Reduced epochs for faster execution
    batch_size=32,
    validation_data=(X_test_lstm, y_test),
    verbose=1
)

print("LSTM model trained.")

# Evaluate the LSTM model
y_pred_lstm_prob = lstm_model.predict(X_test_lstm)
y_pred_lstm = (y_pred_lstm_prob > 0.5).astype(int)

accuracy_lstm = accuracy_score(y_test, y_pred_lstm)
precision_lstm = precision_score(y_test, y_pred_lstm)
recall_lstm = recall_score(y_test, y_pred_lstm)
f1_lstm = f1_score(y_test, y_pred_lstm)

print(f"\n--- LSTM Performance ---")
print(f"Accuracy: {accuracy_lstm:.4f}")
print(f"Precision: {precision_lstm:.4f}")
print(f"Recall: {recall_lstm:.4f}")
print(f"F1 Score: {f1_lstm:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lstm))

print("\nConfusion Matrix:")
cm_lstm = confusion_matrix(y_test, y_pred_lstm)
display(pd.DataFrame(cm_lstm, index=['Actual FAKE', 'Actual REAL'], columns=['Predicted FAKE', 'Predicted REAL']))

Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 827ms/step - accuracy: 0.4297 - loss: 0.6985 - val_accuracy: 0.4688 - val_loss: 0.6967
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 642ms/step - accuracy: 0.5156 - loss: 0.6942 - val_accuracy: 0.4688 - val_loss: 0.6922
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 644ms/step - accuracy: 0.4922 - loss: 0.6940 - val_accuracy: 0.5625 - val_loss: 0.6885
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 924ms/step - accuracy: 0.5469 - loss: 0.6907 - val_accuracy: 0.6562 - val_loss: 0.6852
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 5s 694ms/step - accuracy: 0.5391 - loss: 0.6895 - val_accuracy: 0.6562 - val_loss: 0.6822
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 678ms/step - accuracy: 0.4922 - loss: 0.6894 - val_accuracy: 0.6562 - val_loss: 0.6797
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 646ms/step - accuracy: 0.5078 - loss: 0.6869 - val_accuracy: 0.6562 - val_loss: 0.6766
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 736ms/step - accuracy: 0.5703 - loss: 0.6894 - val_accuracy: 0.6562 - val_loss:

,Predicted FAKE,Predicted REAL
Actual FAKE,5,11
Actual REAL,0,16


## Performance Summary of All Models

In [ ]:
performance_data = {
    'Model': ['SVC', 'Gaussian Naive Bayes', 'CNN', 'LSTM'],
    'Accuracy': [accuracy, accuracy_nb, accuracy_cnn, accuracy_lstm],
    'Precision': [precision, precision_nb, precision_cnn, precision_lstm],
    'Recall': [recall, recall_nb, recall_cnn, recall_lstm],
    'F1 Score': [f1, f1_nb, f1_cnn, f1_lstm]
}

performance_df = pd.DataFrame(performance_data)
display(performance_df)

,Model,Accuracy,Precision,Recall,F1 Score
0,SVC,0.50000,0.500000,0.375,0.428571
1,Gaussian Naive Bayes,0.62500,0.583333,0.875,0.700000
2,CNN,0.62500,0.625000,0.625,0.625000
3,LSTM,0.65625,0.592593,1.000,0.744186
